# Demo — Triaging Medallion Quality Suites & Assigning Remediation Strategies
In Veridian Health's Medallion Lakehouse, automated quality gates validate Bronze and Silver tables across six dimensions (**completeness**, **validity**, **accuracy**, **uniqueness**, **timeliness**, and **consistency**) before data is promoted to Gold clinical and CMS reporting products.

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your AWS Cloud Lab environment and seed the **Veridian Health Medallion Lakehouse** (`veridian_lakehouse` in AWS Glue, S3 `bronze/silver/gold` Parquet prefixes, and AWS Lake Formation enforcement) via `setup_aws.py`.

In [1]:
import os
os.environ["VERIDIAN_LF_LIVE"] = "1"
os.environ["VERIDIAN_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
glue = boto3.client("glue", region_name=os.environ["VERIDIAN_AWS_REGION"])
try:
    glue.get_database(Name="veridian_lakehouse")
    print("Glue database 'veridian_lakehouse' is already seeded and ready.")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

Glue database 'veridian_lakehouse' is already seeded and ready.


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from demo_toolkit import connect_lakehouse, lh_backend

lh = connect_lakehouse()
q = lh.run_medallion_quality()
print(f"Backend: {lh_backend()} -> {type(lh).__name__} | Failing checks: {int((~q['success']).sum())} of {len(q)}")
display(q)

Backend: live -> AwsMedallionLakehouse | Failing checks: 9 of 10


                           table    zone                   column                                       expectation     dimension  success  element_count  unexpected_count  unexpected_percent
0  bronze_lab_device_calibration  bronze           iso15189_grade               expect_column_values_to_not_be_null  completeness    False             30                 4               13.33
1      silver_clinical_encounter  silver               icd10_code               expect_column_values_to_not_be_null  completeness    False            900                53                5.89
2          silver_patient_master  silver                      mrn          expect_column_values_to_match_mrn_format      validity    False            248                14                5.65
3          silver_patient_master  silver                    email         expect_column_values_to_match_email_regex      validity    False            248                 9                3.63
4          silver_patient_master  silver

## 1. Roll up by dimension & inspect Medallion contract SLOs

In [3]:
display(q.groupby("dimension").agg(
    checks=("success", "size"),
    failing=("success", lambda s: int((~s).sum())),
    max_unexpected_pct=("unexpected_percent", "max"),
))
display(lh.medallion_slos())

              checks  failing  max_unexpected_pct
dimension                                        
accuracy           2        2                3.71
completeness       2        2               13.33
consistency        1        1               17.78
timeliness         1        1                2.42
uniqueness         1        0                0.00
validity           3        3                5.65

                           table    zone     dimension                   column                        slo_target                           steward
0  bronze_lab_device_calibration  bronze  completeness           iso15189_grade                 >= 98.0% non-null   diagnostics-lab@veridian.health
1      silver_clinical_encounter  silver  completeness               icd10_code     >= 99.0% primary ICD-10 coded      clinical-ops@veridian.health
2          silver_patient_master  silver      validity                      mrn        >= 99.5% MRN-XXXXXX format  patient-identity@veridian.health
3      

## 2. Assign a remediation strategy to one failing check (`silver_patient_master.mrn`)

When a check fails between Bronze and Silver, we choose between three patterns:
- **`"source"`**: Fix the upstream EHR/device producer or enforce an ingestion schema gate.
- **`"downstream"`**: Filter or impute in the Gold view (e.g., excluding archived enrollments `> 5 years` old).
- **`"quarantine"`**: Route offending rows to a restricted quarantine table before Silver->Gold promotion when bad rows create immediate **HIPAA / GDPR compliance risk** or corrupt **CMS regulatory disclosures**.

In [4]:
mrn_check = q[(q["table"] == "silver_patient_master") & (q["column"] == "mrn")].copy()
mrn_check["remediation_strategy"] = "quarantine"
mrn_check["target_slo"] = ">= 99.5% MRN-XXXXXX format"
display(mrn_check[["table", "zone", "column", "dimension", "unexpected_count", "unexpected_percent", "remediation_strategy", "target_slo"]])

                   table    zone column dimension  unexpected_count  unexpected_percent remediation_strategy                  target_slo
2  silver_patient_master  silver    mrn  validity                14                5.65           quarantine  >= 99.5% MRN-XXXXXX format
